In [ ]:
# =============================================================================
# CELL 1 – IMPORTS, DARK THEME, AND HELPERS
# =============================================================================

import importlib
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown

# -------------------- Dark Theme --------------------
display(HTML("""
<style>
    body, .jp-Notebook, .jp-OutputArea-output, .jp-RenderedHTMLCommon {
        background-color: #1e1e1e !important;
        color: #d4d4d4 !important;
    }
    h2, h3, h4 {
        color: #4fc3f7 !important;
        border-bottom: 2px solid #3498db !important;
        padding-bottom: 4px;
    }
    b, strong { color: #f48fb1 !important; }
    .highlight {
        background-color: #2d2d2d !important;
        padding: 10px;
        border-left: 4px solid #3498db;
        margin: 4px 0;
        color: #d4d4d4;
    }
    code {
        background-color: #333 !important;
        color: #ffcc80 !important;
        padding: 2px 4px;
        border-radius: 4px;
    }
    .dataframe {
        background-color: #2d2d2d !important;
        color: #d4d4d4 !important;
    }
</style>
"""))

# -------------------- Verbosity Flags --------------------
SHOW_VERBOSE = True
SHOW_INFO = True
SHOW_CRITICAL = True
SHOW_DEBUG = True
# For demonstration, we keep them False to reduce output; you can set True as needed.


# -------------------- Import Extraction Module --------------------
import importlib, Extraction as EX
importlib.reload(EX)
DATASETS, CSV_HASHES = EX.discover_processed_datasets(show_info=True)
print(sorted(DATASETS), {k: len(v) for k, v in DATASETS.items()}) # expect amazon_polarity at 100000 rows, and the other 7 datasets at their canonical sizes.

# -------------------- Helper Functions --------------------
def display_title(title: str):
    display(HTML(f"<h2>{title}</h2>"))

def display_info(message: str):
    display(HTML(f"<div class='highlight'>{message}</div>"))
    

def discover_extractions(exp_root: Path) -> pd.DataFrame:
    rows = []
    for f in exp_root.rglob("extraction.json"):
        parts = f.relative_to(exp_root).parts
        try:
            models_idx = parts.index("models")
            datasets_idx = parts.index("datasets")
            model = "/".join(parts[models_idx+1:datasets_idx])
            dataset = parts[datasets_idx+1]
            rows.append({"model": model, "dataset": dataset, "path": str(f)})
        except ValueError:
            rows.append({"model": None, "dataset": None, "path": str(f)})
    return pd.DataFrame(rows)

def load_experiment_results(exp_root: Path) -> pd.DataFrame:
    """
    Load all extraction results from the experiment directory, inferring model and
    dataset names from the file path if they are missing in the JSON.
    """
    records = []
    exp_root = Path(exp_root)
    if not exp_root.exists():
        return pd.DataFrame()

    for meta_file in exp_root.rglob("extraction.json"):
        try:
            with open(meta_file, "r") as f:
                meta = json.load(f)

            # Extract model and dataset from path (most reliable)
            parts = meta_file.relative_to(exp_root).parts
            try:
                models_idx = parts.index("models")
                datasets_idx = parts.index("datasets")
                model = "/".join(parts[models_idx+1:datasets_idx])
                dataset = parts[datasets_idx+1]
            except ValueError:
                # Fallback to metadata
                model = meta.get("model", {}).get("name")
                dataset = meta.get("dataset", {}).get("name")

            record = {
                "experiment_id": meta.get("experiment_id"),
                "model": model,
                "dataset": dataset,
                "status": meta.get("status"),
                "completed_samples": meta.get("performance", {}).get("completed_samples"),
                "total_samples": meta.get("dataset", {}).get("samples"),
                "batch_size": meta.get("extraction", {}).get("batch_size"),
                "pooling": meta.get("extraction", {}).get("pooling"),
                "max_length": meta.get("extraction", {}).get("max_length"),
                "samples_per_second": meta.get("performance", {}).get("samples_per_second"),
                "tokens_per_second": meta.get("performance", {}).get("tokens_per_second"),
                "elapsed_seconds": meta.get("performance", {}).get("elapsed_seconds"),
                "error": None,
                "text_column": meta.get("dataset", {}).get("text_column"),
                "label_column": meta.get("dataset", {}).get("labels", {}).get("label_column"),
            }
            records.append(record)
        except Exception:
            continue

    return pd.DataFrame(records)

print("Environment ready. Dark theme applied.")

In [ ]:
# =============================================================================
# CELL 2 — CANONICAL DATASET DISCOVERY (drop-in, no registry module)
# =============================================================================

from pathlib import Path
import importlib
import Extraction as EX
importlib.reload(EX)

display_title("Processed dataset discovery")

DATASETS, CSV_HASHES = EX.discover_processed_datasets(
    datasets_root=EX.PROCESSED_DATASETS_ROOT,
    show_info=True,
)

display_info(
    f"<b>{len(DATASETS)}</b> processed datasets discovered under "
    f"<code>{EX.PROCESSED_DATASETS_ROOT}</code>"
)

# Visual summary
summary = pd.DataFrame([
    {
        "name": name,
        "rows": len(df),
        "columns": ", ".join(df.columns),
        "csv_sha256": CSV_HASHES[name][:16] + "…",
        "sample_label": str(df["label"].iloc[0]),
    }
    for name, df in DATASETS.items()
])
display(summary)

# Contract checks
for name, df in DATASETS.items():
    assert list(df.columns) == list(EX.PROCESSED_COLUMNS), f"{name}: bad columns"
    assert df.index.is_unique and df.index[0] == 0 and df.index[-1] == len(df) - 1, \
        f"{name}: index must be a clean RangeIndex so row i ↔ hidden_states[i]"
    assert df["label"].map(lambda x: isinstance(x, list) and len(x) > 0).all(), \
        f"{name}: all labels must be non-empty Python lists"

display_info("✅ All datasets satisfy the extraction contract.")

In [ ]:
# =============================================================================
# CELL 2.5 — DETECT PENDING (MODEL, DATASET) PAIRS
# =============================================================================
import numpy as np
from pathlib import Path

def dataset_is_done(model_name: str, dataset_name: str) -> bool:
    """True iff this (model, dataset) folder has a fully-True completed.npy."""
    d = EX.build_dataset_directory(model_name, dataset_name)
    comp = d / "completed.npy"
    meta = d / "extraction.json"
    if not (comp.exists() and meta.exists()):
        return False
    try:
        c = np.load(comp, mmap_mode="r")
        return bool(c.all())
    except Exception:
        return False

def model_is_done(model_name: str) -> bool:
    return all(dataset_is_done(model_name, ds) for ds in DATASETS)

ALL_MODELS = list(EX.ALL_PRIMARY_MODEL_NAMES)
DONE_MODELS    = [m for m in ALL_MODELS if model_is_done(m)]
PENDING_MODELS = [m for m in ALL_MODELS if not model_is_done(m)]

print(f"Total registered : {len(ALL_MODELS)}")
print(f"Complete         : {len(DONE_MODELS)}")
print(f"Pending          : {len(PENDING_MODELS)}")
for m in PENDING_MODELS:
    done_ds = sum(dataset_is_done(m, ds) for ds in DATASETS)
    print(f"  · {m:<55}  {done_ds}/{len(DATASETS)} datasets done")

In [4]:
import importlib
importlib.reload(EX)          # picks up the patched _load_and_apply_order

GATED = {
    "google/gemma-3-270m",
    "google/gemma-3-1b-pt",
    "google/gemma-3-4b-pt",
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-3B",
}
TOO_BIG = {"Qwen/Qwen2.5-3B", "Qwen/Qwen3-1.7B-Base"}   # exceed 8 GiB at fp32

TARGET_MODELS = [m for m in PENDING_MODELS if m not in GATED | TOO_BIG]

print(f"Running {len(TARGET_MODELS)} models")

results = EX.run_experiments( # actual run
    datasets=DATASETS,
    dataset_csv_hashes=CSV_HASHES,
    model_names=TARGET_MODELS,
    pooling="mean",
    max_length=128,
    use_half_precision=True,
    flush_every_batches=8,
    continue_on_model_error=True,
    experiment_id="master_v2",
    show_verbose=True, show_info=True, show_critical=True, show_debug=False,
)


print(f"\nRecords : {len(results)}")


────────────────────────────────────────────────────────────────────────────────────────
RUNTIME / SPEED TELEMETRY :: emotion
────────────────────────────────────────────────────────────────────────────────────────
  Progress             : 5,248/19,999 (26.24%)
  Newly computed       : 5,248
  Wall time            : 32m 26.7s
  Throughput           : 2.70 samples/s
  ETA                  : 1h 31m 11.7s
  Current stage        : measurement

  LAST BATCH
    Range              : 5216:5248
    New samples        : 32
    Total              : 22.64s
    Forward            : 21.66s
    Tokenization       : 0.09s
    Input transfer     : 0.00s
    Pooling            : 0.07s
    Conversion         : 0.00s
    Memmap write       : 0.41s
    Flush              : 0.08s
    Sequence length    : 52
    Tokens             : 631
    Token throughput   : 27.87 tokens/s

  ROLLING PERFORMANCE
    Mean batch         : 15.908s
    Median batch       : 14.212s
    P95 batch          : 24.072s
    Max ba

In [ ]:
# =============================================================================
# CELL 3 — RUN ONLY MODELS WHOSE WEIGHTS ARE ALREADY ON DISK
# =============================================================================

# Two ways to pick the target list. Pick ONE.

# --- Option A: hard-coded to the two downloaded-but-unextracted models ---
TARGET_MODELS = PENDING_MODELS

# --- Option B: every model whose (a) checkpoints exist and (b) extraction pending
# HUB = Path("/Volumes/Amirali/Probing-Emotions/.hf_cache/hub")
# def is_cached(model_name):
#     slug = "models--" + model_name.replace("/", "--")
#     d = HUB / slug
#     return d.is_dir() and any(p.is_dir() for p in (d/"snapshots").iterdir())
# TARGET_MODELS = [m for m in PENDING_MODELS if is_cached(m)]

print("Will run:", TARGET_MODELS)

results = EX.run_experiments( # tests
    datasets=DATASETS,
    dataset_csv_hashes=CSV_HASHES,
    model_names=TARGET_MODELS,      # <-- the key difference from run_model_matrix
    pooling="mean",
    max_length=512,
    use_half_precision=True,
    flush_every_batches=8,
    continue_on_model_error=True,
    experiment_id="master_v1",      # <-- same experiment ID as before: resumes, doesn't restart
    show_verbose=True,
    show_info=True,
    show_critical=True,
    show_debug=True,
)

print(f"\nRecords : {len(results)}")

In [ ]:

# Use the REAL root so .hf_cache/ is reused by the subsequent real run.
# Do NOT redirect EXTERNAL_ROOT here.
EX.configure_external_storage(show_info=True)   # creates .hf_cache/{hub,xet,assets}

# 1 sample per dataset — cheapest way to touch every (model, dataset) pair.
VALIDITY_DATASETS = {name: df.iloc[:1].copy() for name, df in DATASETS.items()}

results = EX.run_model_matrix(                  # groups=None => all 25
    datasets=VALIDITY_DATASETS,
    dataset_csv_hashes=None,                    # see §4 — required for 1-row slices
    groups=None,                                # all 25 models
    experiment_id="run",
    pooling="mean",
    max_length=32,                              # <-- only forward-pass cost knob
    use_half_precision=False,                   # CPU fp32: safest for probing
    flush_every_batches=1,
    continue_on_model_error=True,               # gated/OOM models must not abort the sweep
    show_verbose=True,                         # quiet: 25 models × 4 datasets is noisy
    show_info=True, 
    show_critical=True,
    show_debug=True,
)

print(f"Records       : {len(results)}")
print(f"Cache         : {EX.HF_HUB_CACHE}")
print(f"Cache size GB : {sum(f.stat().st_size for f in EX.HF_HUB_CACHE.rglob('*') if f.is_file())/1e9:.2f}")

In [ ]:
""" Main run of the models - full datasets. 
# =============================================================================
# CELL 3 — RUN MODEL MATRIX ON CLEAN DATA
# =============================================================================

# The pipeline writes to a single fixed root:
#     /Volumes/Amirali/Probing-Emotions/
# `base_output` and `auto_batch_size` no longer exist as arguments.
# Ordering is controlled by /Volumes/Amirali/Probing-Emotions/extraction_order.json.

NEW_EXPERIMENT_ID = "master_v1"

results = EX.run_model_matrix(
    datasets=DATASETS,
    dataset_csv_hashes=CSV_HASHES,
    groups=None,
    experiment_id=NEW_EXPERIMENT_ID,
    pooling="mean",
    max_length=512,
    use_half_precision=True,
    flush_every_batches=8,
    continue_on_model_error=True,
    show_verbose=True,
    show_info=True,
    show_critical=True,
    show_debug=True,
)

print(f"Result records : {len(results)}")
print(f"Output root    : {EX.EXTERNAL_ROOT}")
print(f"HF cache       : {EX.HF_HUB_CACHE}")
"""

### How to load the missing QWENs ?  

In [ ]:
# =============================================================================
# CELL 4 — SUMMARY FROM RUNS/CLEAN_V1
# =============================================================================
EXTERNAL_ROOT = "/Volumes/Amirali/Probing-Emotions"
RUN_ROOT = EXTERNAL_ROOT / "runs" / NEW_EXPERIMENT_ID
df_results = load_experiment_results(RUN_ROOT)

if not df_results.empty:
    display_title("Extraction progress")
    df_show = df_results.copy()
    df_show["model"] = df_show["model"].str.replace("__", "/", regex=False)
    display(df_show[["model", "dataset", "status", "completed_samples", "total_samples"]])
    display_info(f"Pairs discovered: <b>{len(df_results)}</b>")
else:
    display_info("No extraction metadata yet — run CELL 3.")

In [ ]:
from sys import audit


df_audit = pd.DataFrame(audit)
if not df_audit.empty:
    # Primary contract: extraction's n_samples must equal the processed CSV's
    # row count. Otherwise the memmap is misaligned with the source data.
    def _csv_len(dataset_name):
        return len(DATASETS.get(dataset_name, []))
    df_audit["csv_rows"] = df_audit["dataset_name"].map(_csv_len)
    df_audit["csv_matches"] = df_audit["n_samples"] == df_audit["csv_rows"]
    df_audit["row_count_ok"] = df_audit["n_samples"] == df_audit["completed_count"]

    display(df_audit[[
        "model_name", "dataset_name", "n_samples", "csv_rows", "completed_count",
        "csv_matches", "row_count_ok",
        "status", "checksum_match", "sample_ids_match",
    ]])

In [ ]:
# =============================================================================
# CELL 5 – VISUALISATIONS & ANALYSIS 
# =============================================================================

if not df_results.empty:
    # Prepare data for plotting
    df = df_results.copy()
    df['completion_pct'] = df['completed_samples'] / df['total_samples'] * 100
    df['status_clean'] = df['status'].replace({'complete': 'Complete', 'partial': 'Partial', 'failed': 'Failed', 'already_complete': 'Already Complete'})

    sns.set_style("darkgrid")
    plt.rcParams.update({
        'figure.facecolor': '#1e1e1e',
        'axes.facecolor': '#2d2d2d',
        'axes.edgecolor': '#d4d4d4',
        'axes.labelcolor': '#d4d4d4',
        'text.color': '#d4d4d4',
        'xtick.color': '#d4d4d4',
        'ytick.color': '#d4d4d4',
        'grid.color': '#444444',
        'legend.facecolor': '#2d2d2d',
        'legend.edgecolor': '#d4d4d4',
    })

    # ---- 1. Completion status per model/dataset ----
    fig, ax = plt.subplots(figsize=(12, 8))
    pivot = df.pivot_table(index='model', columns='dataset', values='completion_pct', aggfunc='max')
    sns.heatmap(pivot, annot=True, fmt=".0f", cmap="viridis", cbar_kws={'label': 'Completion %'}, ax=ax)
    ax.set_title('Completion Percentage by Model and Dataset', color='#4fc3f7')
    plt.tight_layout()
    plt.show()

    # ---- 2. Throughput (samples/sec) by model ----
    df_complete = df[df['samples_per_second'].notna()]
    if not df_complete.empty:
        fig, ax = plt.subplots(figsize=(14, 6))
        sns.barplot(data=df_complete, x='model', y='samples_per_second', hue='dataset', palette='coolwarm', ax=ax)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.set_title('Extraction Throughput (samples/sec)', color='#4fc3f7')
        plt.tight_layout()
        plt.show()

    # ---- 3. Total time per model ----
    df_time = df.groupby('model')['elapsed_seconds'].sum().reset_index().sort_values('elapsed_seconds', ascending=False)
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(data=df_time, x='elapsed_seconds', y='model', palette='magma', ax=ax)
    ax.set_xlabel('Total Elapsed Time (seconds)')
    ax.set_title('Cumulative Extraction Time per Model', color='#4fc3f7')
    plt.tight_layout()
    plt.show()

    # ---- 4. Label coverage for completed datasets ----
    # We'll just display label column info
    display_title("Label Columns Used")
    display(df[['model', 'dataset', 'label_column']].drop_duplicates())
else:
    display_info("No data to visualise. Please run extraction first.")

In [ ]:
# =============================================================================
# CELL 6 – FINAL SUMMARY & EXPORT
# =============================================================================

if not df_results.empty:
    # Save consolidated CSV
    output_csv = exp_root / "extraction_summary.csv"
    df_results.to_csv(output_csv, index=False)
    display_title("Final Report")
    display(df_results)
    display_info(f"Report exported to <code>{output_csv}</code>")
else:
    display_info("Nothing to export yet.")

In [ ]:
discover_extractions(exp_root)

In [ ]:
# Check which models have at least one dataset complete
model_status = df_results.groupby("model")["dataset"].nunique()
print(model_status)

Model Availability on DevNeeds.ir

Cross-referenced against your 25-model MODEL_REGISTRY. All models ≤ 1.5B fit your Mac's memory envelope; the Qwen2-7B failure confirms a roughly 3B ceiling in fp16.

✅ Directly Available — 6 of 25

Exact name matches. No changes to MODEL_REGISTRY required.

Model	Params
google-bert/bert-base-uncased	110M
distilbert/distilbert-base-uncased	66M
microsoft/deberta-v3-small	140M
Qwen/Qwen2.5-0.5B	500M
Qwen/Qwen2.5-1.5B	1.5B
meta-llama/Llama-3.2-1B	1B
⚠️ Available as Close Variants — 5 of 25

Functionally equivalent, different training regime. MODEL_REGISTRY names would need editing.

Your name	DevNeeds substitute	Difference
Qwen/Qwen3-0.6B-Base	Qwen/Qwen3-0.6B	post-trained
meta-llama/Llama-3.2-3B	meta-llama/Llama-3.2-3B-Instruct	instruction-tuned
google/gemma-3-1b-pt	google/gemma-3-1b-it	instruction-tuned
google/gemma-3-4b-pt	google/gemma-3-4b-it	instruction-tuned
google/gemma-3-270m	google/gemma-3-270m-it	instruction-tuned
❌ Not Available on DevNeeds — 14 of 25

Absent entirely. Requires Cloudflare Worker, foreign download, or manual transfer.

Model	Missing family
FacebookAI/roberta-base	RoBERTa
google/electra-small-discriminator	ELECTRA
gpt2	GPT-2
EleutherAI/gpt-neo-125m	GPT-Neo
facebook/opt-125m	OPT
HuggingFaceTB/SmolLM2-135M	SmolLM2
HuggingFaceTB/SmolLM2-360M	SmolLM2
HuggingFaceTB/SmolLM2-1.7B	SmolLM2
Qwen/Qwen2-0.5B	Qwen2 (superseded)
Qwen/Qwen2-1.5B	Qwen2 (superseded)
Qwen/Qwen2.5-3B	Qwen2.5
Qwen/Qwen3-1.7B-Base	Qwen3
Qwen/Qwen3-4B-Base	Qwen3
TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T	TinyLlama
Impact: Three entire families — RoBERTa, ELECTRA, and the GPT-2/GPT-Neo/OPT trio — are the canonical baselines in emotion-probing literature. Their absence materially weakens the model matrix unless resolved through a different route.

✨ Recommended Additions From DevNeeds

All fit your machine. Ranked by value for hidden-state probing.

Encoders — every layer meaningfully distinct

Model	Params
albert/albert-base-v2	12M
albert/albert-large-v2	18M
google-bert/bert-base-cased	110M
google-bert/bert-base-multilingual-cased	177M
google-bert/bert-large-uncased	335M
google-bert/bert-large-cased	335M
huggingface/distilbert-base-uncased-finetuned-mnli	66M
Encoder–decoder — different architectural family

Model	Params
google/flan-t5-small	60M
google/flan-t5-base	220M
Small modern decoders

Model	Params
Qwen/Qwen2.5-0.5B-Instruct	500M
Qwen/Qwen2.5-Coder-0.5B	500M
Qwen/Qwen1.5-0.5B-Chat	500M
Qwen/Qwen3-0.6B	600M
meta-llama/Llama-3.2-1B-Instruct	1B
meta-llama/Llama-Guard-3-1B	1B
meta-llama/Prompt-Guard-86M	86M
google/embeddinggemma-300m	300M
Sentence-transformers — embedding-specialised

Model	Params
sentence-transformers/all-MiniLM-L6-v2	22M
sentence-transformers/paraphrase-MiniLM-L6-v2	22M
sentence-transformers/all-mpnet-base-v2	110M
sentence-transformers/all-distilroberta-v1	82M
sentence-transformers/LaBSE	471M
Multilingual seq2seq

Model	Params
facebook/m2m100_418M	418M
facebook/nllb-200-distilled-600M	600M
Multimodal (optional)

Model	Params
openai/clip-vit-base-patch32	150M
openai/whisper-tiny	39M
openai/whisper-base	74M
Notes

DevNeeds is not a uniform API. Entries carry inconsistent suffixes (/main, /revision, commit SHAs). Downloads arrive as folders; you must manually reconstruct the cache layout under .hf_cache/hub/models--<owner>--<name>/snapshots/main/.
The Cloudflare Worker remains the stronger solution. One deployment unlocks the entire Group 3 list in a single endpoint; DevNeeds structurally cannot deliver those families.
Trim MODEL_REGISTRY before running. A focused 12-model matrix drawn from Groups 1 and 2 is scientifically stronger than 25 entries where 14 fail at download.